# Official TDC DTI benchmark — pehli baar apna binding model duniya se tulega

**Ye notebook kyun zaroori hai.** ADMET ke 22 benchmark humne official protocol pe
chalaye the, isliye "top models ke barabar" kehna sach hai. Par **binding hamesha
apne hi splits pe naapa** — kisi se tulna nahi hui. Bina iske "doosron se behtar"
kehna jhooth hoga, aur ek technical buyer pehle 10 minute me pakad lega.

**Benchmark:** TDC `DTI_DG_Group` → **BindingDB_Patent**, metric **PCC**
(Pearson correlation). Ye *domain generalization* task hai — purane patents pe
train, naye patents pe test. Yaani asli duniya jaisa: aane wale naye molecules pe
kaam karna.

**Official leaderboard (28 May 2026):**

| Rank | Model | PCC |
|---|---|---|
| 1 | Otter-Knowledge-Ensemble | 0.588 |
| 2 | **OrangeBalls-ProtBertMorgan** | **0.538** |
| 3 | MMD | 0.433 |
| 4 | CORAL | 0.432 |
| 5 | **ERM** (baseline) | **0.427** |
| 6 | MTL | 0.425 |
| 7 | GroupDRO | 0.384 |
| 8 | AndMASK | 0.288 |
| 9 | IRM | 0.284 |

Rank 2 pe dhyaan do: **ProtBert + Morgan** — ye lagbhag apna hi architecture hai
(protein language model + Morgan fingerprint). Isliye ek **imaandaar target**:
pehle **ERM (0.427) ko chhuo ya haraao**, phir 0.538 ki taraf dekho.

Jo nikle wahi sach hai — kam aaye to kam likhenge. **Yahi number buyer ko
dikhane layak hai**, apne splits wala CI nahi.

⛔ **Runtime → Change runtime type → T4 GPU** (TPU nahi). Phir **Run all**.
Pehli baar **1–2 ghante** (ESM embeddings). Sab Drive me cache hoga, aur beech
me disconnect ho jaye to dobara chalane pe wahi se aage chalega.

## 1. Tools

In [ ]:
import subprocess, sys
def pip(*a):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a],
                       capture_output=True, text=True)
    print(("✅ " if r.returncode == 0 else "❌ ") + " ".join(a))
    if r.returncode: print(r.stderr[-1200:])
pip("rdkit"); pip("transformers"); pip("xgboost"); pip("joblib"); pip("scikit-learn")
# PyTDC bina dependencies ke - taaki Colab ka numpy/pandas/fsspec na badle.
# (Seedha "pip install PyTDC" fsspec upgrade kar deta hai aur phir Colab
#  "restart session" maangta hai - pehle wahi dikkat aa chuki hai.)
pip("--no-deps", "PyTDC")
pip("fuzzywuzzy", "huggingface_hub", "tqdm", "seaborn", "requests")

In [ ]:
import os, json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from transformers import AutoTokenizer, AutoModel
from scipy.stats import pearsonr
RDLogger.DisableLog("rdApp.*")
np.random.seed(0); torch.manual_seed(0)

IS_TPU = bool(os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_NAME"))
if IS_TPU:
    print("⛔ TPU chuna hua hai - T4 GPU chahiye.")
    print("   Runtime → Change runtime type → T4 GPU → Save, phir Run all")
GPU = torch.cuda.is_available(); DEVICE = "cuda" if GPU else "cpu"
print("Ready ✅ |", "GPU mila" if GPU else "⚠️ GPU nahi - ye notebook CPU pe bahut dheema chalega")
try:
    from tdc.benchmark_group import dti_dg_group
    print("PyTDC ✅")
except Exception as e:
    raise SystemExit(f"PyTDC import nahi hua: {type(e).__name__}: {e}\n"
                     "Agar 'restart session' maange to Runtime → Restart session, phir Run all.")

## 2. Settings

In [ ]:
SEEDS      = 5        # official protocol 5 seeds maangta hai
N_TREES    = 1200
MAX_PROT   = 20000    # itne se zyada unique proteins hue to sabse zyada data wale
# Checkpoint ka size feature ke hisaab se alag hai. Ek hi bada size rakhna galti
# thi: ESM sabse dheema hai, aur agar proteins 5,000 se kam hue to pehla
# checkpoint SAB khatam hone ke baad hi lagta - yaani beech me disconnect pe
# poora ESM ka kaam zaaya. Isliye ESM ke liye chhota chunk.
CKPT_RDKIT = 5000     # ~5 ms per molecule, tez
CKPT_CHEM  = 5000     # ChemBERTa chhota model, tez
CKPT_ESM   = 200      # ESM-2 650M sabse dheema - har 200 proteins pe save
ESM_MODEL  = "facebook/esm2_t33_650M_UR50D"
CHEM_MODEL = "DeepChem/ChemBERTa-77M-MTR"

## 3. Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/ai_drug_platform"
OUT_DIR = os.path.join(ROOT, "tdc_dti_dg"); os.makedirs(OUT_DIR, exist_ok=True)
print("Drive ready ✅", OUT_DIR)

## 4. Official benchmark data
TDC khud hi split deta hai — hum split ko chhute bhi nahi. Yahi iska matlab hai:
jo number aayega wo **usi paimane pe** hoga jis pe leaderboard ke models hain.

In [ ]:
group = dti_dg_group(path=os.path.join(OUT_DIR, "tdc_data"))
BM = "BindingDB_Patent"
b = group.get(BM)
train_val, test = b["train_val"].reset_index(drop=True), b["test"].reset_index(drop=True)
print(f"{BM}: train_val {len(train_val):,} · test {len(test):,}")
print("columns:", list(train_val.columns))
print(f"\nunique drugs : train_val {train_val.Drug.nunique():,} · test {test.Drug.nunique():,}")
print(f"unique targets: train_val {train_val.Target.nunique():,} · test {test.Target.nunique():,}")
new_t = set(test.Target) - set(train_val.Target)
new_d = set(test.Drug) - set(train_val.Drug)
print(f"\ntest me naye targets: {len(new_t):,} · naye drugs: {len(new_d):,}")
print("(domain generalization = test me purane se alag patents, isliye naya bahut hoga)")
print(f"\nY: min {train_val.Y.min():.2f} max {train_val.Y.max():.2f} median {train_val.Y.median():.2f}")

## 5. Features — v2 wali hi recipe
Molecule = Morgan FP 1024 + RDKit descriptors + ChemBERTa · Protein = ESM-2 650M

Har feature set Drive pe **checkpoint** hota hai. ESM sabse dheema hai, isliye
uska checkpoint **har 200 proteins** pe lagta hai (molecules ke liye 5,000 pe).
Colab disconnect ho jaye to dobara **Run all** karne pe wahi se aage chalega —
cell `⏯️ ... pehle se ho chuka - wahi se aage chalega` print karega.

In [ ]:
fp1024 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
DESC = Descriptors._descList
print(f"RDKit {len(DESC)} descriptors")

_MDL = {}
def _get(name, half=False):
    if name not in _MDL:
        tok = AutoTokenizer.from_pretrained(name)
        mdl = AutoModel.from_pretrained(name).to(DEVICE).eval()
        if half and GPU: mdl = mdl.half()
        _MDL[name] = (tok, mdl)
    return _MDL[name]

def rdkit_feats(smis):
    out, bad, w = [], 0, 1024 + len(DESC)
    for s in smis:
        m = Chem.MolFromSmiles(str(s))
        if m is None:
            out.append(np.zeros(w, np.float32)); bad += 1; continue
        d = []
        for _, fn in DESC:
            try: d.append(float(fn(m)))
            except Exception: d.append(np.nan)
        d = np.nan_to_num(np.clip(np.array(d, np.float64), -1e6, 1e6)).astype(np.float32)
        out.append(np.concatenate([fp1024.GetFingerprintAsNumPy(m).astype(np.float32), d]))
    if bad: print(f"  {bad} molecules padhe nahi gaye (zero kar diye)")
    return np.stack(out)

@torch.no_grad()
def chemberta(smis, batch=256):
    tok, mdl = _get(CHEM_MODEL)
    out = []
    for i in range(0, len(smis), batch):
        t = tok([str(x) for x in smis[i:i+batch]], return_tensors="pt", padding=True,
                truncation=True, max_length=256).to(DEVICE)
        h = mdl(**t).last_hidden_state
        m = t["attention_mask"].unsqueeze(-1).float()
        out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

@torch.no_grad()
def esm(seqs, window=1000):
    tok, mdl = _get(ESM_MODEL, half=True)
    out = []
    for s in seqs:
        s = str(s).upper().strip()
        chunks = [s[i:i+window] for i in range(0, len(s), window)] or [s]
        vs = []
        for c in chunks:
            t = tok(c, return_tensors="pt").to(DEVICE)
            vs.append(mdl(**t).last_hidden_state[0, 1:-1].float().mean(0).cpu().numpy())
        out.append(np.mean(vs, 0))
    return np.stack(out).astype(np.float32)

def cached(name, items, build, chunk):
    p = os.path.join(OUT_DIR, name + ".npy"); part = p.replace(".npy", ".part.npy")
    if os.path.exists(p):
        a = np.load(p)
        if len(a) == len(items):
            print(f"  Drive se load: {name} {a.shape}"); return a
        print(f"  ⚠️ {name}: purana cache ({len(a):,}) ab ke data ({len(items):,}) se mel nahi khata")
        os.remove(p)
    acc = None
    if os.path.exists(part):
        try:
            acc = np.load(part)
            if len(acc) > len(items): acc = None; os.remove(part)
            else:
                print(f"  ⏯️ {name}: {len(acc):,}/{len(items):,} pehle se ho chuka "
                      f"- wahi se aage chalega (zero se nahi)")
        except Exception: acc = None
    t0, done0 = time.time(), (0 if acc is None else len(acc))
    while acc is None or len(acc) < len(items):
        i = 0 if acc is None else len(acc)
        a = build(items[i:i+chunk])
        acc = a if acc is None else np.vstack([acc, a])
        np.save(part, acc)
        el = time.time() - t0
        rate = (len(acc) - done0) / max(el, 1e-9)
        left = (len(items) - len(acc)) / rate / 60 if rate > 0 else 0
        print(f"  💾 {name} {len(acc):,}/{len(items):,}  ({el/60:.1f} min beeta · ~{left:.0f} min baaki)")
    np.save(p, acc)
    try: os.remove(part)
    except Exception: pass
    print(f"  bana: {name} {acc.shape} ({(time.time()-t0)/60:.1f} min)")
    return acc

In [ ]:
ALL = pd.concat([train_val, test], ignore_index=True)
DRUGS = sorted(ALL.Drug.astype(str).unique())
PROTS = sorted(ALL.Target.astype(str).unique())
if len(PROTS) > MAX_PROT:
    keep = set(ALL.Target.value_counts().head(MAX_PROT).index)
    PROTS = sorted(p for p in PROTS if p in keep)
    print(f"⚠️ proteins {MAX_PROT:,} tak seemit kiye")
print(f"{len(DRUGS):,} unique molecules · {len(PROTS):,} unique proteins\n")

DF = cached("drug_rdkit", DRUGS, rdkit_feats, CKPT_RDKIT)
DC = cached("drug_chemberta", DRUGS, chemberta, CKPT_CHEM)
PE = cached("prot_esm650", PROTS, esm, CKPT_ESM)
for _k in list(_MDL): del _MDL[_k]
if GPU: torch.cuda.empty_cache()

DRUG_X = np.hstack([DF, DC]).astype(np.float32); del DF, DC
PROT_X = PE.astype(np.float32); del PE
DI = {s: i for i, s in enumerate(DRUGS)}
PI = {s: i for i, s in enumerate(PROTS)}
print(f"\nDrug features {DRUG_X.shape} · Protein features {PROT_X.shape} ✅")
print(f"Memory: {(DRUG_X.nbytes + PROT_X.nbytes)/1e9:.2f} GB")

## 6. Train aur official evaluation
5 seeds, har seed pe TDC ka apna train/valid split. Prediction TDC ko wapas dete
hain aur **wahi** PCC nikalta hai — hum khud metric nahi gin rahe.

In [ ]:
import xgboost as xgb

def XY(df):
    d = df.Drug.astype(str).map(DI)
    p = df.Target.astype(str).map(PI)
    ok = d.notna() & p.notna()
    if (~ok).any(): print(f"    {int((~ok).sum()):,} rows chhoote (feature nahi mila)")
    d, p = d[ok].astype(int).values, p[ok].astype(int).values
    X = np.hstack([DRUG_X[d], PROT_X[p]]).astype(np.float32)
    return X, df.Y.values[ok.values], ok.values

preds = []
for seed in range(1, SEEDS + 1):
    t0 = time.time()
    tr, va = group.get_train_valid_split(benchmark=BM, split_type="default", seed=seed)
    Xtr, ytr, _ = XY(tr)
    m = xgb.XGBRegressor(n_estimators=N_TREES, max_depth=8, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.6, tree_method="hist",
                         device=DEVICE, n_jobs=os.cpu_count(), random_state=seed)
    m.fit(Xtr, ytr)
    del Xtr
    Xte, yte, ok = XY(test)
    p_ok = m.predict(Xte); del Xte
    # jo rows chhoot gaye unke liye train ka median - TDC poori test list maangta hai
    full = np.full(len(test), float(np.median(ytr)), dtype=np.float32)
    full[ok] = p_ok
    preds.append({BM: full})
    r = pearsonr(yte, p_ok)[0]
    print(f"  seed {seed}: apna hisaab PCC={r:.4f}  ({(time.time()-t0)/60:.1f} min)")

results = group.evaluate_many(preds)
print("\n=== OFFICIAL TDC RESULT ===")
print(results)

## 7. Leaderboard se tulna — **iska screenshot Claude ko bhejna hai**

In [ ]:
LB = [("Otter-Knowledge-Ensemble", 0.588), ("OrangeBalls-ProtBertMorgan", 0.538),
      ("MMD", 0.433), ("CORAL", 0.432), ("ERM (baseline)", 0.427), ("MTL", 0.425),
      ("GroupDRO", 0.384), ("AndMASK", 0.288), ("IRM", 0.284)]
# TDC result ki key LOWERCASE me deta hai ("bindingdb_patent"), isliye seedha
# results[BM] KeyError deta hai. Case-insensitive dhundte hain, aur value list
# ya dict dono roop sambhal lete hain.
print("results ka asli roop:", results)
key = BM if BM in results else next((k for k in results if k.lower() == BM.lower()), None)
if key is None:
    key = list(results)[0]
    print(f"⚠️ '{BM}' nahi mila, '{key}' use kar rahe hain")
val = results[key]
if isinstance(val, dict):
    val = list(val.values())[0]
if isinstance(val, (list, tuple)):
    ours = float(val[0]); ours_sd = float(val[1]) if len(val) > 1 else 0.0
else:
    ours = float(val); ours_sd = 0.0

rows = [{"model": n, "PCC": v, "kaun": "leaderboard"} for n, v in LB]
rows.append({"model": "APNA (v2 recipe)", "PCC": round(ours, 4), "kaun": "hum"})
tb = pd.DataFrame(rows).sort_values("PCC", ascending=False).reset_index(drop=True)
tb.index += 1
print(f"APNA SCORE: PCC = {ours:.4f} ± {ours_sd:.4f}\n")
print(tb.to_string())

rank = int(tb[tb.kaun == "hum"].index[0])
beat = [n for n, v in LB if ours > v]
print(f"\nRank: {rank} / {len(tb)}")
print(f"Inse behtar: {', '.join(beat) if beat else 'kisi se nahi'}")
erm = 0.427
if ours > 0.538:   verdict = "🎉 ProtBertMorgan se bhi aage - ye bikne layak number hai"
elif ours > erm:   verdict = "✅ ERM baseline se behtar - imaandaar dawa kar sakte hain"
elif ours > 0.384: verdict = "⚠️ ERM se neeche, par GroupDRO se upar - sudhaar chahiye"
else:              verdict = "❌ leaderboard me neeche - abhi 'behtar' kehna jhooth hoga"
print(f"\n{verdict}")

os.makedirs(OUT_DIR, exist_ok=True)
json.dump({"benchmark": key, "metric": "PCC", "ours": ours, "ours_std": ours_sd,
           "rank": rank, "beat": beat, "leaderboard": dict(LB), "verdict": verdict,
           "n_seeds": SEEDS, "esm": ESM_MODEL, "chem": CHEM_MODEL},
          open(os.path.join(OUT_DIR, "tdc_dti_dg_result.json"), "w"), indent=2)
print(f"\nsave: {OUT_DIR}/tdc_dti_dg_result.json")

**Kaise padhein.** Ye number apne splits wale CI se bahut kam lagega, aur wo
theek hai — ye **mushkil** benchmark hai (naye patents pe test), aur yahan sab
models 0.28–0.59 me hain. Jo bhi aaye, **wahi** sach hai aur wahi kisi buyer ko
dikhane layak hai.

Agla kadam uske baad: **pocket-aware representation** — poore sequence ke bajaye
kinase ke 85 pocket residues pe ESM. Us se ye number badhna chahiye.